<a href="https://colab.research.google.com/github/RDGopal/IB9LQ-2026/blob/main/MLM9_Business_Utility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#MLM9 — Business Utility

This notebook trains a compact CNN to classify post-disaster property damage and frames evaluation as a downstream business utility question.

## Data setup
Upload `xbd_teaching_subset.zip` to the current Colab runtime. Expected contents after extraction:

```text
xbd_teaching_subset/
  labels.csv
  pre_images/
  post_images/
```

This cell sets up the data by checking for the zipped dataset, extracting it if necessary, and then loading the `labels.csv` file into a pandas DataFrame. It also displays basic information about the loaded labels.

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
ZIP_PATH = Path('xbd_teaching_subset.zip')
DATA_DIR = Path('xbd_teaching_subset')
if not ZIP_PATH.exists():
    matches = list(Path.cwd().rglob('xbd_teaching_subset.zip'))
    if matches:
        ZIP_PATH = matches[0]
if not DATA_DIR.exists():
    assert ZIP_PATH.exists(), 'Upload xbd_teaching_subset.zip to the Colab runtime, then rerun this cell.'
    print('Extracting:', ZIP_PATH.resolve())
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('.')
else:
    print('Using existing extracted folder:', DATA_DIR.resolve())
if not (DATA_DIR/'labels.csv').exists() and Path('labels.csv').exists():
    DATA_DIR = Path('.')
labels = pd.read_csv(DATA_DIR/'labels.csv')
print('DATA_DIR:', DATA_DIR.resolve())
print('Rows:', len(labels))
display(labels.head())
display(labels.damage_label.value_counts())
display(pd.crosstab(labels.split, labels.damage_label))

This cell imports essential libraries for data manipulation, image processing, deep learning with PyTorch, and plotting. It also defines global constants like the device (CPU/GPU), sets a random seed for reproducibility, defines mappings for damage labels, and includes utility functions for loading images and displaying image grids.

In [ ]:
import math, random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(42)
DAMAGE_TO_ID={'no-damage':0,'minor-damage':1,'major-damage':2,'destroyed':3}
ID_TO_DAMAGE={v:k for k,v in DAMAGE_TO_ID.items()}
def load_image(path, size=64):
    img=Image.open(path).convert('RGB').resize((size,size))
    arr=np.array(img).astype('float32')/255.0
    arr=arr*2.0-1.0
    return torch.from_numpy(arr).permute(2,0,1)
def show_grid(xs, titles=None, ncols=4, figsize=(10,6)):
    xs=xs.detach().cpu().clamp(-1,1); xs=(xs+1)/2
    n=len(xs); ncols=min(ncols,n); nrows=math.ceil(n/ncols)
    plt.figure(figsize=figsize)
    for i in range(n):
        plt.subplot(nrows,ncols,i+1); plt.imshow(xs[i].permute(1,2,0).numpy()); plt.axis('off')
        if titles is not None: plt.title(titles[i], fontsize=8)
    plt.tight_layout(); plt.show()

## Dataset

This cell defines the `PostDamageDataset` class, a custom PyTorch Dataset for loading images and their corresponding damage labels. It then initializes the training, validation, and test datasets and dataloaders, and displays a sample of images from the training set with their labels.

In [ ]:
class PostDamageDataset(Dataset):
    def __init__(self,data_dir,labels,split='train',image_size=64):
        self.data_dir=Path(data_dir); self.df=labels[labels['split'].eq(split)].reset_index(drop=True); self.image_size=image_size
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        r=self.df.iloc[idx]; return load_image(self.data_dir/r.post_image,self.image_size), int(r.damage_id)
IMAGE_SIZE=64; BATCH_SIZE=64
train_ds=PostDamageDataset(DATA_DIR,labels,'train',IMAGE_SIZE); val_ds=PostDamageDataset(DATA_DIR,labels,'val',IMAGE_SIZE); test_ds=PostDamageDataset(DATA_DIR,labels,'test',IMAGE_SIZE)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=0); val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0); test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
print('Train:',len(train_ds),'Val:',len(val_ds),'Test:',len(test_ds))
x,y=next(iter(train_loader)); show_grid(x[:8],[ID_TO_DAMAGE[int(c)] for c in y[:8]],ncols=4,figsize=(10,5))

## Class balance and weights

This cell analyzes the class balance within the training data and calculates class weights. These weights are used in the loss function to mitigate the impact of class imbalance during model training.

In [ ]:
display(pd.crosstab(labels['split'],labels['damage_label']))
class_counts=labels[labels['split'].eq('train')]['damage_id'].value_counts().sort_index(); class_weights=torch.ones(4)
for cls_id in range(4):
    if cls_id in class_counts.index: class_weights[cls_id]=len(train_ds)/(4*class_counts.loc[cls_id])
class_weights=class_weights.to(DEVICE); print('Class weights:',class_weights.detach().cpu().numpy())

## Model

This cell defines the `DamageCNN` class, which is a compact Convolutional Neural Network (CNN) for image classification. It specifies the architecture of the neural network and then initializes an instance of the model and sets up the AdamW optimizer for training.

In [ ]:
class DamageCNN(nn.Module):
    def __init__(self,num_classes=4):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(128,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(128,num_classes))
    def forward(self,x): return self.net(x)
model=DamageCNN().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)

## Train and evaluate

This cell defines the `evaluate` function to assess the model's performance on a given dataset (e.g., validation or test set) by calculating loss, accuracy, and predictions. It also defines `per_class_accuracy` to show performance for each damage category. The main training loop then iterates for a specified number of epochs, performing forward and backward passes, updating weights, and evaluating the model on the validation set after each epoch. Finally, it plots the training history (loss and accuracy over epochs).

In [ ]:
def evaluate(model,loader):
    model.eval(); total=correct=0; loss_sum=0; ys=[]; ps=[]
    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(DEVICE),y.to(DEVICE); logits=model(x); loss=F.cross_entropy(logits,y,weight=class_weights); pred=logits.argmax(1)
            total+=y.numel(); correct+=(pred==y).sum().item(); loss_sum+=loss.item()*y.numel(); ys.append(y.cpu()); ps.append(pred.cpu())
    return {'loss':loss_sum/max(total,1),'accuracy':correct/max(total,1),'y_true':torch.cat(ys).numpy() if ys else np.array([]),'y_pred':torch.cat(ps).numpy() if ps else np.array([])}
def per_class_accuracy(y_true,y_pred):
    return pd.DataFrame([{'damage_class':ID_TO_DAMAGE[i],'n':int((y_true==i).sum()),'accuracy':np.nan if (y_true==i).sum()==0 else float((y_pred[y_true==i]==i).mean())} for i in range(4)])
EPOCHS=50; history=[]
for epoch in range(1,EPOCHS+1):
    model.train(); losses=[]
    for x,y in train_loader:
        x,y=x.to(DEVICE),y.to(DEVICE); loss=F.cross_entropy(model(x),y,weight=class_weights)
        opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
    val=evaluate(model,val_loader); history.append({'epoch':epoch,'train_loss':float(np.mean(losses)),'val_loss':val['loss'],'val_accuracy':val['accuracy']})
    print(f"Epoch {epoch:03d} | train_loss={np.mean(losses):.4f} | val_loss={val['loss']:.4f} | val_acc={val['accuracy']:.3f}")
hist=pd.DataFrame(history); display(hist)
plt.figure(figsize=(6,4)); plt.plot(hist.epoch,hist.train_loss,label='train loss'); plt.plot(hist.epoch,hist.val_loss,label='validation loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.title('Training curve'); plt.show()
plt.figure(figsize=(6,4)); plt.plot(hist.epoch,hist.val_accuracy,label='validation accuracy'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.ylim(0,1); plt.legend(); plt.title('Validation accuracy'); plt.show()

## Test-set evaluation

This cell evaluates the trained model on the unseen test set, reporting the overall test loss and accuracy. It also displays a detailed per-class accuracy breakdown and a confusion matrix to visualize the model's performance across different damage categories.

In [ ]:
test=evaluate(model,test_loader); print('Test loss:',round(test['loss'],4)); print('Test accuracy:',round(test['accuracy'],3))
y_true=test['y_true']; y_pred=test['y_pred']; display(per_class_accuracy(y_true,y_pred))
cm=pd.crosstab(pd.Series(y_true).map(ID_TO_DAMAGE),pd.Series(y_pred).map(ID_TO_DAMAGE),rownames=['Actual'],colnames=['Predicted'],dropna=False)
class_names=[ID_TO_DAMAGE[i] for i in range(4)]; cm=cm.reindex(index=class_names,columns=class_names,fill_value=0); display(cm)
plt.figure(figsize=(6,5)); plt.imshow(cm.values); plt.xticks(range(4),cm.columns,rotation=45,ha='right'); plt.yticks(range(4),cm.index); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Damage Classification Confusion Matrix')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]): plt.text(j,i,int(cm.values[i,j]),ha='center',va='center')
plt.tight_layout(); plt.show()

## Inspect predictions

This cell performs an inspection of individual predictions from the test set. It separates correctly classified and incorrectly classified examples and then displays a grid of selected incorrect predictions, showing both the true and predicted damage labels to highlight where the model struggles.

In [ ]:
model.eval(); examples=[]
with torch.no_grad():
    for x,y in test_loader:
        pred=model(x.to(DEVICE)).argmax(1).cpu()
        for i in range(len(x)): examples.append((x[i],int(y[i]),int(pred[i])))
        if len(examples)>=64: break
wrong=[e for e in examples if e[1]!=e[2]]; right=[e for e in examples if e[1]==e[2]]
print('Correct:',len(right),'Incorrect:',len(wrong)); chosen=(wrong[:8] if wrong else right[:8])
xs=torch.stack([e[0] for e in chosen]); titles=[f"true={ID_TO_DAMAGE[e[1]]}\npred={ID_TO_DAMAGE[e[2]]}" for e in chosen]
show_grid(xs,titles,ncols=4,figsize=(10,5))

## Summary and Business Implications

In this notebook, we developed and evaluated a compact CNN model for classifying post-disaster property damage into four categories: no-damage, minor-damage, major-damage, and destroyed. We framed the evaluation not just on technical metrics like accuracy, but also considered the downstream business utility of such a model.

**Key takeaways:**

*   **Data Setup and Preprocessing:** We set up the XBD teaching subset, loaded image labels, and addressed class imbalance by calculating and applying class weights to the loss function during training.
*   **Model Training and Evaluation:** A `DamageCNN` model was trained for 50 epochs, and its performance was monitored on a validation set. The training curves showed the model learning, though with some fluctuations in validation metrics.
*   **Test-Set Performance:** The final model was evaluated on an unseen test set, providing overall accuracy and a per-class breakdown. The confusion matrix visually highlighted where the model performed well and where it struggled (e.g., misclassifying 'destroyed' as 'minor-damage').
*   **Prediction Inspection:** By inspecting individual correct and incorrect predictions, we gained insights into the types of errors the model makes.

**Business Utility and Implications:**

*   **Faster Damage Assessment:** A well-performing damage classification model can significantly speed up the initial assessment of properties after a disaster, allowing insurance companies, relief organizations, and local authorities to prioritize resources more effectively.
*   **Resource Allocation:** Accurate damage labels can guide the allocation of adjusters, repair crews, and aid to the most affected areas, reducing response times and improving efficiency.
*   **Cost Savings:** By automating or semi-automating the initial damage assessment, organizations can reduce operational costs associated with manual inspections.
*   **Improved Customer Experience:** Faster claims processing and support can lead to higher satisfaction among affected individuals.
*   **Challenges and Future Work:** While the model provides a good starting point, the test set results and inspection of incorrect predictions suggest areas for improvement. For example, further fine-tuning, exploring more advanced architectures, or incorporating pre-disaster imagery could enhance accuracy, particularly for distinguishing between severe damage categories. The class imbalance was addressed with weighting, but further techniques like augmentation or synthetic data generation might yield better results for underrepresented classes.

#Required Task 10
##Visual Risk Scorecard for Property Underwriting
Your task is to build a Visual Risk Scorecard for Property Underwriting using the `xbd_teaching_subset` dataset and the damage-classification model developed in MLM9.

**Objective:** Move beyond simple classification accuracy and convert model predictions into a business-oriented risk score and underwriting or claims action.

###Instructions

**Load the Test Data**

•	Use the same processed xBD dataset used in MLM7-MLM9.

•	From `labels.csv`, select only the records where `split == "test"`.

•	Store these records in a new DataFrame, for example `df_test_properties`.

•	Each record should include at least: `post_image, damage_label, and damage_id`.

Example:

`df_test_properties = labels[labels["split"].eq("test")].copy()`


**Load or Train the Damage Classifier**

•	Use the CNN damage classifier from MLM9.

•	You may either load your saved trained model from MLM9, if available, or retrain the MLM9 classifier using the training split.

•	The classifier should output predicted probabilities for the four damage classes: `no-damage, minor-damage, major-damage`, and `destroyed`.

**Generate Predicted Class Probabilities**

•	Iterate through each property image.

•	For each post-disaster image, load the image from post_images/.

•	Run the image through the trained classifier.

•	Convert the model outputs into class probabilities using softmax.

•	Store the probabilities in new columns: `p_no_damage`, `p_minor_damage`, `p_major_damage` and `p_destroyed`.

•	Also store the predicted class label in a new column called `predicted_damage_label`.

**Compute an Expected Damage Score**

•	Create a single numerical damage score for each property using the damage scale below.

•	For each image, compute the expected damage score as the probability-weighted average of the damage classes.

•	Add this value as a new column called `expected_damage_score`. The score should range approximately from 0 to 3.


**Damage scale:**
Damage class	Score

no-damage	-> 0

minor-damage ->	1

major-damage ->	2

destroyed	-> 3

**Expected damage score formula:**

`expected_damage_score = (0 * p_no_damage + 1 * p_minor_damage + 2 * p_major_damage + 3 * p_destroyed)`

**Assign an Underwriting or Claims Action**

•	Use expected_damage_score to assign a recommended business action.

•	Add the recommended action as a new column called `recommended_action`.

**Expected damage score	Recommended action**

< 0.75	`Auto-clear`

0.75 to < 1.75	`Human review`

\>= 1.75	`Urgent inspection / reserve adjustment`

**Evaluate the Risk Scorecard**

•	Compare the model predictions with the ground-truth damage labels.

•	Calculate and report the following: `overall classification accuracy`; `confusion matrix`; `average expected_damage_score` by true `damage_label`; `number of properties assigned to each recommended_action`; `percentage of truly major-damage or destroyed properties` assigned to `Urgent inspection / reserve adjustment`; `percentage of no-damage properties assigned to Auto-clear`.

•	These metrics help evaluate whether the scorecard is useful for business decision-making, not just whether the model is statistically accurate.

**Display Results**

•	Display a sample of `df_test_properties` with the columns listed below.
•	Display a summary table showing `recommended_action`, `number_of_properties`, and `percentage_of_test_set`.


**Sample output columns:**

`post_image, damage_label, predicted_damage_label, p_no_damage, p_minor_damage, p_major_damage, p_destroyed, expected_damage_score, recommended_action`


**Short Written Interpretation**

•	Write a brief interpretation of your results.

•	Address whether the scorecard separates low-risk and high-risk properties reasonably well.

•	Identify which damage classes are most often confused.

•	Discuss whether the action threshold is too conservative or too lenient.

•	Explain whether you would trust this model for full automation, and why or why not.
